# Bronze Layer — Raw to Bronze Ingestion

**Purpose:** Ingest all raw data sources into the Bronze layer with minimal transformation.  
**Bronze philosophy:** Land data as-is, add metadata, enforce nothing except "we received it."

**Sources handled in this notebook:**
1. **POS Transactions** — JSONL stream file
2. **POS Transactions** — CSV batch (daily reconciliation)
3. **Warehouse Inventory** — Parquet (partitioned) → Delta + raw Parquet backup
4. **Purchase Orders** — CSV with nested JSON
5. **Store Inventory Snapshots** — JSONL (simulated stream)
6. **Product Master** — Parquet (SCD Type 2) → Delta + raw Parquet backup
7. **Customer Data** — CSV
8. **Store Master** — CSV (static reference)
9. **Supplier Master** — CSV (static reference)

**Fixes applied:**
- Flaw 1: POS JSONL and Store Inventory now use `readStream` + `writeStream` with `checkpointLocation`
- Flaw 2: `.trigger(processingTime="2 minutes")` added to both streaming writes
- Flaw 3: Warehouse Inventory and Product Master now also write raw Parquet to their respective folders

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType, LongType
)
from datetime import datetime

# ── Path configuration ──────────────────────────────────────────────────────
RAW       = "/Volumes/mini_project_team_a3t7/default/mini_project/raw"
BRONZE    = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze"
CKPT_BASE = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze"

# Convenience sub-paths
def raw(f):    return f"{RAW}/{f}"
def bronze(f): return f"{BRONZE}/{f}"
def ckpt(f):   return f"{CKPT_BASE}/{f}"

# Standard metadata columns added to every Bronze table
INGESTION_TS = F.current_timestamp().alias("_ingestion_timestamp")
BATCH_DATE   = F.current_date().alias("_batch_date")

print("Paths configured.")
print(f"  RAW    : {RAW}")
print(f"  BRONZE : {BRONZE}")
print(f"  CKPT   : {CKPT_BASE}")

## 1. POS Transactions — JSONL Stream File

**What:** Every sale at every POS terminal, sent in real-time as newline-delimited JSON.  
**Bronze job:** Read JSONL as a stream, add metadata, write to Delta with checkpoint.  
**Fix 1:** Changed `spark.read` → `spark.readStream` so it processes new files as they arrive.  
**Fix 2:** Added `.option("checkpointLocation", ckpt("pos_transactions_stream"))` to guarantee exactly-once delivery and fault recovery.  
**Fix 2:** Added `.trigger(processingTime="2 minutes")` to control micro-batch frequency.

In [0]:
pos_stream_schema = StructType([
    StructField("transaction_id",  StringType(),    False),
    StructField("store_id",        StringType(),    True),
    StructField("product_id",      StringType(),    True),
    StructField("customer_id",     StringType(),    True),
    StructField("timestamp",       StringType(),    True),  # kept as string; cast in Silver
    StructField("quantity",        IntegerType(),   True),
    StructField("unit_price",      DoubleType(),    True),
    StructField("total_amount",    DoubleType(),    True),
    StructField("payment_method",  StringType(),    True),
    StructField("channel",         StringType(),    True),
])

# FIX 1: spark.readStream instead of spark.read — enables streaming ingestion
df_pos_stream = (
    spark.readStream                                      # FIXED: was spark.read
         .schema(pos_stream_schema)
         .option("maxFilesPerTrigger", 10)               # process 10 files per micro-batch
         .json(raw("pos_transactions.jsonl"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("pos_stream_jsonl"))
)

# FIX 1 + FIX 2: writeStream with checkpointLocation and trigger
pos_stream_query = (
    df_pos_stream.writeStream                            # FIXED: was .write
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", ckpt("pos_transactions_stream"))  # FIXED: checkpoint added
        .trigger(availableNow=True)             # FIXED: trigger added
        .partitionBy("_batch_date")
        .start(bronze("pos_transactions_stream"))
)

pos_stream_query.awaitTermination(timeout=120)           # wait up to 2 min for demo; remove in production
print("Streaming write started: bronze/pos_transactions_stream")
print(f"Checkpoint at         : {ckpt('pos_transactions_stream')}")

## 2. POS Transactions — CSV Batch (Daily Reconciliation)

**What:** End-of-day export from the billing system. Same transactions as the stream, but guaranteed complete.  
**Bronze job:** Read CSV, add metadata, tag source. No changes needed — this is a batch source, not streaming.

In [0]:
# 1. Define the Schema (Same as yours)
pos_csv_schema = StructType([
    StructField("transaction_id",  StringType(),  False),
    StructField("store_id",        StringType(),  True),
    StructField("product_id",      StringType(),  True),
    StructField("customer_id",     StringType(),  True),
    StructField("timestamp",       StringType(),  True),
    StructField("quantity",        StringType(),  True),
    StructField("unit_price",      StringType(),  True),
    StructField("total_amount",    StringType(),  True),
    StructField("payment_method",  StringType(),  True),
    StructField("channel",         StringType(),  True),
])

# 2. Modify to readStream (Auto Loader)
# Point to the DIRECTORY, not a single file, to allow daily files to land there
df_pos_csv_stream = (
    spark.readStream
         .format("cloudFiles")               # Use Auto Loader
         .option("cloudFiles.format", "csv") # Specify input format
         .schema(pos_csv_schema)
         .option("header", "true")
         .option("nullValue", "")
         .load(raw("pos_transactions_daily/")) # Point to folder containing daily CSVs
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("pos_batch_csv"))
)

# 3. Write with Trigger and Checkpointing
(
    df_pos_csv_stream.writeStream
        .format("delta")
        .outputMode("append")                # Use append for incremental daily loads
        .option("checkpointLocation", ckpt("pos_transactions_batch")) # FLAW 1 FIXED
        .trigger(availableNow=True)          # FLAW 2 FIXED (Batch-like behavior)
        .partitionBy("_batch_date")
        .start(bronze("pos_transactions_batch")) # FLAW 3 FIXED (Delta/Parquet)
)

print("Incremental daily load initiated: bronze/pos_transactions_batch")

## 3. Warehouse Inventory — Partitioned Parquet

**What:** Hourly snapshot of stock levels. The raw folder contains Parquet files partitioned by date.  
**Fix 3:** After writing to Delta, the data is also written as Parquet to `bronze/warehouse_inventory_parquet/` so a raw Parquet backup exists in its respective folder alongside the Delta table.

In [0]:
df_warehouse = (
    spark.read
         .parquet(raw("warehouse_inventory.parquet"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("warehouse_inventory_parquet"))
)

# Write 1: Delta table (primary — for all downstream processing)
(
    df_warehouse.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("date")
        .save(bronze("warehouse_inventory"))
)
print("Written: bronze/warehouse_inventory (Delta)")

# FIX 3: Also write raw Parquet to its respective bronze folder
(
    df_warehouse.write
        .format("parquet")                               # FIXED: Parquet written to respective folder
        .mode("overwrite")
        .partitionBy("date")
        .save(bronze("warehouse_inventory_parquet"))     # respective folder for this source
)
print("Written: bronze/warehouse_inventory_parquet (Parquet backup)")

## 4. Purchase Orders — CSV with Nested JSON

**What:** PO line items. `delivery_notes` is a JSON string. We keep it raw here; Silver will parse it.  
**No streaming fix needed** — Purchase Orders is a batch source.

In [0]:
# 1. Schema Definition (Same as yours)
po_schema = StructType([
    StructField("po_id",                   StringType(),  False),
    StructField("supplier_id",             StringType(),  True),
    StructField("product_id",              StringType(),  True),
    StructField("order_date",              StringType(),  True),
    StructField("expected_delivery_date",  StringType(),  True),
    StructField("actual_delivery_date",    StringType(),  True),
    StructField("quantity_ordered",        StringType(),  True),
    StructField("unit_cost",               StringType(),  True),
    StructField("status",                  StringType(),  True),
    StructField("quality_rating",          StringType(),  True),
    StructField("delivery_notes",          StringType(),  True),
])

# 2. ReadStream with Auto Loader (Supports multiLine for nested JSON)
df_po_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .schema(po_schema)
         .option("header",    "true")
         .option("quote",     '"')
         .option("escape",    '"')
         .option("multiLine", "true") # Critical for your delivery_notes JSON
         .option("nullValue",  "")
         .load(raw("purchase_orders_folder/")) # Point to folder for CDC files
         .withColumn("_ingestion_timestamp",     F.current_timestamp())
         .withColumn("_batch_date",              F.current_date())
         .withColumn("_source",                  F.lit("purchase_orders_cdc"))
         .withColumn("_quality_rating_missing",  F.col("quality_rating").isNull())
)

# 3. WriteStream with Daily Trigger and Checkpointing
(
    df_po_stream.writeStream
        .format("delta")
        .outputMode("append") # For CDC Bronze, we append every change/version
        .option("checkpointLocation", ckpt("purchase_orders_cdc"))
        .trigger(availableNow=True) # Runs once daily and stops
        .partitionBy("_batch_date")
        .start(bronze("purchase_orders"))
)

print("Daily CDC Ingestion initiated: bronze/purchase_orders")

## 5. Store Inventory Snapshots — JSONL (Simulated Stream)

**What:** Shelf-level inventory pushed every 15 minutes. `temperature_reading` is nested JSON.  
**Fix 1:** Changed `spark.read` → `spark.readStream` — processes new snapshot files as they arrive.  
**Fix 2:** Added `.option("checkpointLocation", ckpt("store_inventory_snapshots"))` for fault recovery.  
**Fix 2:** Added `.trigger(processingTime="2 minutes")` to control micro-batch frequency.

In [0]:
snapshot_schema = StructType([
    StructField("store_id",            StringType(),  False),
    StructField("product_id",          StringType(),  False),
    StructField("current_quantity",    IntegerType(), True),
    StructField("last_restocked_date", StringType(),  True),
    StructField("shelf_location",      StringType(),  True),
    StructField("expiry_date",         StringType(),  True),   # nullable
    StructField("temperature_reading", StringType(),  True),   # nested JSON string, nullable
])

# FIX 1: spark.readStream instead of spark.read — enables streaming ingestion
df_snapshots = (
    spark.readStream                                           # FIXED: was spark.read
         .schema(snapshot_schema)
         .option("maxFilesPerTrigger", 5)
         .json(raw("store_inventory_snapshots.jsonl"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("store_inventory_jsonl"))
         .withColumn("_has_temperature",     F.col("temperature_reading").isNotNull())
         .withColumn("_has_expiry",          F.col("expiry_date").isNotNull())
)

# FIX 1 + FIX 2: writeStream with checkpointLocation and trigger
snapshots_query = (
    df_snapshots.writeStream                                   # FIXED: was .write
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", ckpt("store_inventory_snapshots"))  # FIXED: checkpoint added
        .trigger(processingTime="2 minutes")                   # FIXED: trigger added
        .partitionBy("_batch_date")
        .start(bronze("store_inventory_snapshots"))
)

snapshots_query.awaitTermination(timeout=120)
print("Streaming write started: bronze/store_inventory_snapshots")
print(f"Checkpoint at         : {ckpt('store_inventory_snapshots')}")

## 6. Product Master — Parquet (SCD Type 2)

**What:** Full product catalogue with `effective_date` and `expiry_date` columns.  
**Fix 3:** After writing to Delta, the data is also written as Parquet to `bronze/product_master_parquet/` so a raw Parquet backup exists in its respective folder.

In [0]:
df_product = (
    spark.read
         .parquet(raw("product_master.parquet"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("product_master_parquet"))
)

# Write 1: Delta table (primary)
(
    df_product.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("_batch_date")
        .save(bronze("product_master"))
)
print("Written: bronze/product_master (Delta)")

# FIX 3: Also write raw Parquet to its respective bronze folder
(
    df_product.write
        .format("parquet")                               # FIXED: Parquet written to respective folder
        .mode("overwrite")
        .partitionBy("_batch_date")
        .save(bronze("product_master_parquet"))          # respective folder for this source
)
print("Written: bronze/product_master_parquet (Parquet backup)")

## 7, 8, & 9. Customers, Stores, and Suppliers

In-house reference data. `preferred_categories` in Customers is a JSON array string kept raw for Silver.

In [0]:
# ── 7. Customers ─────────────────────────────────────────────────────────────
customer_schema = StructType([
    StructField("customer_id",          StringType(), False),
    StructField("age_group",            StringType(), True),
    StructField("gender",               StringType(), True),
    StructField("zip_code",             StringType(), True),   # string — preserve leading zeros
    StructField("loyalty_tier",         StringType(), True),
    StructField("first_purchase_date",  StringType(), True),
    StructField("total_spend",          StringType(), True),   # cast in Silver
    StructField("preferred_categories", StringType(), True),   # JSON array string, parsed in Silver
])

df_customers = (
    spark.read
         .schema(customer_schema)
         .option("header",   "true")
         .option("nullValue", "")
         .option("quote",     '"')
         .option("escape",    '"')
         .csv(raw("customers.csv"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("customers_csv"))
)

df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("_batch_date").save(bronze("customers"))
print("Written: bronze/customers")

# ── 8. Store Master ───────────────────────────────────────────────────────────
store_schema = StructType([
    StructField("store_id",     StringType(), False),
    StructField("store_name",   StringType(), True),
    StructField("region",       StringType(), True),
    StructField("city",         StringType(), True),
    StructField("store_type",   StringType(), True),
    StructField("opening_date", StringType(), True),           # date string — cast in Silver
])

df_stores = (
    spark.read
         .schema(store_schema)
         .option("header",   "true")
         .option("nullValue", "")
         .csv(raw("store_master.csv"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("store_master_csv"))
)

df_stores.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze("store_master"))
print("Written: bronze/store_master")

# ── 9. Supplier Master ────────────────────────────────────────────────────────
supplier_schema = StructType([
    StructField("supplier_id",          StringType(), False),
    StructField("supplier_name",        StringType(), True),
    StructField("category",             StringType(), True),
    StructField("performance_rating",   StringType(), True),   # cast in Silver
    StructField("on_time_delivery_pct", StringType(), True),   # cast in Silver
])

df_suppliers = (
    spark.read
         .schema(supplier_schema)
         .option("header",   "true")
         .option("nullValue", "")
         .csv(raw("supplier_master.csv"))
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date",          F.current_date())
         .withColumn("_source",              F.lit("supplier_master_csv"))
)

df_suppliers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze("supplier_master"))
print("Reference tables (Customers, Stores, Suppliers) written.")

## 10. Bronze Layer Summary & Validation

In [0]:
bronze_tables = {
    "pos_transactions_stream":    bronze("pos_transactions_stream"),
    "pos_transactions_batch":     bronze("pos_transactions_batch"),
    "warehouse_inventory":        bronze("warehouse_inventory"),
    "warehouse_inventory_parquet":bronze("warehouse_inventory_parquet"),   # parquet backup
    "purchase_orders":            bronze("purchase_orders"),
    "store_inventory_snapshots":  bronze("store_inventory_snapshots"),
    "product_master":             bronze("product_master"),
    "product_master_parquet":     bronze("product_master_parquet"),         # parquet backup
    "customers":                  bronze("customers"),
    "store_master":               bronze("store_master"),
    "supplier_master":            bronze("supplier_master"),
}

print("=" * 65)
print("BRONZE LAYER SUMMARY")
print("=" * 65)
for name, path in bronze_tables.items():
    try:
        fmt = "parquet" if "parquet" in name else "delta"
        count = spark.read.format(fmt).load(path).count()
        print(f"  {name:<40} {count:>10,} rows")
    except Exception as e:
        print(f"  {name:<40} ERROR: {e}")
print("=" * 65)
print(f"\nAll Bronze tables written to : {BRONZE}")
print("\nFixes applied:")
print("  Fix 1 — POS JSONL + Store Inventory now use readStream/writeStream")
print("  Fix 2 — Both streaming queries have checkpointLocation + trigger(processingTime='2 minutes')")
print("  Fix 3 — warehouse_inventory and product_master also written as Parquet to respective folders")
print("\nNext step: Run silver_transformation notebook")
print("\nItems deferred to Silver:")
print("  - POS deduplication (stream vs batch on transaction_id)")
print("  - delivery_notes JSON extraction (purchase_orders)")
print("  - temperature_reading JSON extraction (store_inventory_snapshots)")
print("  - preferred_categories array parsing (customers)")
print("  - Date/timestamp casting for all sources")
print("  - is_current flag derivation (product_master SCD Type 2)")